In [1]:
pip install pycryptodome

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\lepha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import tkinter as tk
from tkinter import messagebox
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
import binascii

# Khóa bí mật cố định cho demo (32 bytes cho AES-256)
# Trong thực tế, khóa này sẽ do phần của bạn Trường (Diffie-Hellman) tạo ra
SECRET_KEY = b'12345678901234567890123456789012'

class VinhApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Vinh - AES Encryption Demo")
        self.root.geometry("500x450")

        # 1. Nhập tin nhắn
        tk.Label(root, text="Nhập tin nhắn (Bản rõ):", font=('Arial', 10, 'bold')).pack(pady=5)
        self.txt_plain = tk.Entry(root, width=50)
        self.txt_plain.pack(pady=5)

        # Nút bấm mã hóa
        tk.Button(root, text="Mã hóa tin nhắn", command=self.handle_encrypt, bg="blue", fg="white").pack(pady=5)

        # 2. Hiển thị dữ liệu đã mã hóa (Nhiệm vụ xử lý lưu trữ)
        tk.Label(root, text="Dữ liệu đã mã hóa (Lưu trữ/Truyền đi):", font=('Arial', 10, 'bold')).pack(pady=5)
        self.txt_cipher = tk.Text(root, height=4, width=50, bg="#f0f0f0")
        self.txt_cipher.pack(pady=5)

        # 3. Kết quả giải mã (Nhiệm vụ hiển thị)
        tk.Button(root, text="Giải mã tin nhắn", command=self.handle_decrypt, bg="green", fg="white").pack(pady=5)
        tk.Label(root, text="Kết quả sau giải mã (Bản rõ):", font=('Arial', 10, 'bold')).pack(pady=5)
        self.lbl_result = tk.Label(root, text="...", fg="darkgreen", font=('Arial', 11, 'italic'))
        self.lbl_result.pack(pady=10)

        # Biến tạm để lưu dữ liệu mã hóa (mô phỏng bộ nhớ đệm)
        self.last_encrypted_data = None

    def handle_encrypt(self):
        plain_text = self.txt_plain.get()
        if not plain_text:
            messagebox.showwarning("Chú ý", "Vui lòng nhập tin nhắn!")
            return

        # Khởi tạo AES chế độ CBC
        cipher = AES.new(SECRET_KEY, AES.MODE_CBC)
        iv = cipher.iv # Vector khởi tạo
        
        # Padding và Mã hóa
        ct_bytes = cipher.encrypt(pad(plain_text.encode('utf-8'), AES.block_size))
        
        # Chuyển sang Hex để dễ nhìn và lưu trữ
        hex_iv = binascii.hexlify(iv).decode()
        hex_ct = binascii.hexlify(ct_bytes).decode()
        
        self.last_encrypted_data = (hex_iv, hex_ct)
        
        # Hiển thị lên giao diện
        self.txt_cipher.delete(1.0, tk.END)
        self.txt_cipher.insert(tk.END, f"IV: {hex_iv}\nCipher: {hex_ct}")

    def handle_decrypt(self):
        if not self.last_encrypted_data:
            messagebox.showwarning("Lỗi", "Chưa có dữ liệu để giải mã!")
            return

        try:
            hex_iv, hex_ct = self.last_encrypted_data
            iv = binascii.unhexlify(hex_iv)
            ct = binascii.unhexlify(hex_ct)
            
            # Khởi tạo AES để giải mã
            cipher = AES.new(SECRET_KEY, AES.MODE_CBC, iv=iv)
            pt = unpad(cipher.decrypt(ct), AES.block_size).decode('utf-8')
            
            self.lbl_result.config(text=pt)
        except Exception as e:
            messagebox.showerror("Lỗi", f"Không thể giải mã: {e}")

if __name__ == "__main__":
    root = tk.Tk()
    app = VinhApp(root)
    root.mainloop()